# Source Confound Analysis.

Quantify how much of the reported classification performance is attributable to
the acquisition source rather than to pathology.

The corpus draws Pneumonia entirely from CheXpert, while Normal and Tuberculosis
contain none. Class membership is therefore partly predictable from acquisition
characteristics alone, and a model could achieve high accuracy by identifying the
dataset of origin. The submitted paper acknowledged this but defended against it
only by observing that the compact architecture performs worse than the
backbones, which does not address the question: a weak model on a confounded task
is still solving a confounded task.

Reviewer 5 asked for a source-discriminating classifier. This notebook provides
that, together with two further tests that the request implies.

#### Rationale:
- Train a classifier to predict acquisition source rather than diagnosis, so that
  the separability of the sources is measured rather than assumed.
- Distinguish the global source test, which is entangled with class by
  construction, from the within-class test, which is not.
- Evaluate the trained checkpoints on a subset in which source is held constant,
  giving a performance estimate free of the confound.
- Reuse the existing patient-grouped splits throughout, so that every measurement
  inherits the verified disjointness of the clean corpus.

## 1. Setup.

Configure paths and device, following notebooks 01 to 05.

#### Rationale:
- Keep path handling identical across the pipeline.
- Reuse the clean corpus splits rather than constructing new ones.
- Detect the compute device automatically.
- Fail early if the corpus or its provenance column is missing.

In [1]:
import sys, json, math, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             confusion_matrix, precision_recall_fscore_support)

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR      = PROJECT_ROOT / "data"
EXPERIMENTS   = PROJECT_ROOT / "experiments"
CHECKPOINTS   = EXPERIMENTS / "checkpoints"
POOLS_DIR     = EXPERIMENTS / "pools"
SPLITS_DIR    = EXPERIMENTS / "pools_clean"
TABLE_DIR     = PROJECT_ROOT / "reports" / "tables"
FIG_DIR       = PROJECT_ROOT / "reports" / "figures"
for d in (TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

cfg = json.loads((POOLS_DIR / "preprocessing_config.json").read_text())
IMG_SIZE       = int(cfg["img_size"])
CLASS_TO_INDEX = {k: int(v) for k, v in cfg["class_to_index"].items()}
INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}
IMAGENET_MEAN, IMAGENET_STD = cfg["imagenet_mean"], cfg["imagenet_std"]
CXR_MEAN, CXR_STD           = cfg["cxr_mean"], cfg["cxr_std"]

print(f"Device: {DEVICE.upper()} | image size {IMG_SIZE}")

Device: CUDA | image size 256


## 2. Corpus and Source Composition.

Load the clean corpus splits and set out the relationship between class and
acquisition source, which determines what each test can and cannot establish.

Pneumonia is drawn wholly from CheXpert and neither of the other classes contains
any CheXpert image. A classifier asked to separate CheXpert from the remainder is
therefore also separating Pneumonia from the remainder, and a high score would
demonstrate nothing beyond that construction. Normal and Tuberculosis each span
three sources, so within either class the source can be varied while the
diagnosis is held fixed. That is the test that carries information.

#### Rationale:
- State the composition explicitly, since it governs the interpretation of every
  figure that follows.
- Reuse the existing split assignment for the source task, so that disjointness
  by path, content hash and patient group is inherited rather than re-established.
- Report the class balance of each source task, since the within-class tasks are
  markedly imbalanced and accuracy alone would mislead.
- Resolve absolute paths through the same source key used by the dataset class.

In [2]:
frames = []
for split in ("train", "val", "test"):
    f = pd.read_csv(SPLITS_DIR / f"{split}_split.csv")
    f["split"] = split
    frames.append(f)
corpus = pd.concat(frames, ignore_index=True)

if len(corpus) != 3216:
    raise ValueError(f"Unexpected corpus size {len(corpus)}, expected 3,216.")
if "source_true" not in corpus.columns:
    raise KeyError("source_true column missing; provenance cannot be recovered.")

CHEXPERT_PATH = DATA_DIR / "chexpert"
TBX11K_PATH   = DATA_DIR / "tbx11k"

def resolve_path(row) -> Path:
    if row["Source"] == "chexpert":
        return CHEXPERT_PATH / row["rel_path"]
    return TBX11K_PATH / "imgs" / row["rel_path"]

corpus["resolved"] = corpus.apply(resolve_path, axis=1)

print("Class by source:")
print(pd.crosstab(corpus["source_true"], corpus["Class"]).to_string())

print()
print("Pneumonia is 100% CheXpert; Normal and TB contain none. The global source")
print("task is therefore entangled with class. Within Normal and within TB the")
print("source varies while the diagnosis does not, which is the informative test.")

print()
print("Within-class source composition:")
for cls in ("Normal", "TB"):
    sub = corpus[corpus["Class"] == cls]
    tbx = int((sub["source_true"] == "tbx11k").sum())
    oth = len(sub) - tbx
    print(f"  {cls:8s} tbx11k {tbx:4d} | other (Montgomery, Shenzhen, DA, DB) {oth:4d}")

Class by source:
Class        Normal  Pneumonia   TB
source_true                        
chexpert          0       1072    0
da+db            30          0   74
mc+shenzhen      57          0  198
tbx11k          985          0  800

Pneumonia is 100% CheXpert; Normal and TB contain none. The global source
task is therefore entangled with class. Within Normal and within TB the
source varies while the diagnosis does not, which is the informative test.

Within-class source composition:
  Normal   tbx11k  985 | other (Montgomery, Shenzhen, DA, DB)   87
  TB       tbx11k  800 | other (Montgomery, Shenzhen, DA, DB)  272


## 3. Pixel-Level Source Signature.

Before training anything, test whether the source leaves a signature in the raw
pixels. Images are reduced to 64x64, z-scored to remove brightness and contrast,
and compared by correlation.

The comparison is made **within** a class. Two Normal radiographs from the same
source are compared against two Normal radiographs from different sources. If the
same-source pairs are systematically more similar, the acquisition source is
visible in the pixels independently of the diagnosis.

An earlier version of this analysis compared same-class against same-source pairs
across the whole corpus. That comparison is uninformative: the two categories
overlap by construction, since every same-class Pneumonia pair is necessarily a
same-source pair. The within-class formulation avoids this.

#### Rationale:
- Establish whether a signature exists before attributing model behaviour to it.
- Hold the diagnosis fixed so that any difference is attributable to source alone.
- Use a measurement that requires no training and therefore no further modelling
  assumptions.
- Report both classes separately, since they differ in how many images come from
  the smaller sources.

In [3]:
def load_zscored(paths, side=64):
    """Load images as z-scored unit vectors for correlation."""
    out = np.zeros((len(paths), side * side), dtype=np.float32)
    for i, p in enumerate(paths):
        arr = np.asarray(Image.open(p).convert("L").resize((side, side),
                         Image.Resampling.LANCZOS), dtype=np.float32).ravel()
        arr = arr - arr.mean()
        n = np.linalg.norm(arr)
        out[i] = arr / n if n > 0 else arr
    return out

print("Within-class source signature (Pearson correlation at 64x64):")
print()
signature_rows = []

for cls in ("Normal", "TB"):
    sub = corpus[corpus["Class"] == cls].reset_index(drop=True)
    X = load_zscored(sub["resolved"].tolist())
    sim = X @ X.T
    iu = np.triu_indices(len(sub), k=1)
    vals = sim[iu]

    src = sub["source_true"].values
    same = src[iu[0]] == src[iu[1]]

    m_same, m_diff = vals[same].mean(), vals[~same].mean()
    # Welch t-test without scipy dependency on unequal variances.
    v1, v2 = vals[same].var(ddof=1), vals[~same].var(ddof=1)
    n1, n2 = same.sum(), (~same).sum()
    t = (m_same - m_diff) / math.sqrt(v1 / n1 + v2 / n2)

    print(f"  {cls}:")
    print(f"    same source      mean {m_same:.4f}  (n={n1:,} pairs)")
    print(f"    different source mean {m_diff:.4f}  (n={n2:,} pairs)")
    print(f"    difference       {m_same - m_diff:+.4f}   t = {t:+.1f}")
    signature_rows.append({"class": cls, "same_source_mean": round(m_same, 4),
                           "diff_source_mean": round(m_diff, 4),
                           "difference": round(m_same - m_diff, 4),
                           "t_statistic": round(t, 1),
                           "n_same": int(n1), "n_diff": int(n2)})

pd.DataFrame(signature_rows).to_csv(TABLE_DIR / "source_pixel_signature.csv", index=False)
print()
print("A positive difference means images sharing an acquisition source resemble")
print("one another more than images of the same diagnosis from different sources.")

Within-class source signature (Pearson correlation at 64x64):

  Normal:
    same source      mean 0.7114  (n=486,651 pairs)
    different source mean 0.6193  (n=87,405 pairs)
    difference       +0.0922   t = +153.1
  TB:
    same source      mean 0.6040  (n=341,804 pairs)
    different source mean 0.5744  (n=232,252 pairs)
    difference       +0.0295   t = +67.2

A positive difference means images sharing an acquisition source resemble
one another more than images of the same diagnosis from different sources.


## 4. Source Classifier: Dataset and Training Routine.

Define the machinery for training a classifier whose target is the acquisition
source rather than the diagnosis.

The existing split assignment is reused unchanged. Only the label differs, so the
disjointness already verified for the clean corpus, by file path, content hash and
patient group, carries over without further argument.

#### Rationale:
- Reuse the verified splits rather than constructing new ones, so leakage cannot
  be reintroduced through a second splitting procedure.
- Use a pretrained backbone, since the question is whether the source is
  detectable at all rather than how efficiently it can be learned.
- Train briefly, because a source signal strong enough to matter will be evident
  within a few epochs.
- Report balanced accuracy and AUROC, since the within-class tasks are imbalanced
  and plain accuracy would be dominated by the majority source.

In [4]:
class SourceDataset(Dataset):
    """Images labelled by acquisition source instead of diagnosis."""

    def __init__(self, frame: pd.DataFrame, label_col: str, transform):
        self.items = frame.reset_index(drop=True)
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row = self.items.iloc[idx]
        img = Image.open(row["resolved"]).convert("RGB")
        return self.transform(img), torch.tensor(int(row[self.label_col]),
                                                 dtype=torch.long)


from src.transforms import build_shared_transforms
train_tf, eval_tf = build_shared_transforms(img_size=IMG_SIZE,
                                            mean=IMAGENET_MEAN, std=IMAGENET_STD)


def train_source_classifier(frame: pd.DataFrame, label_col: str, title: str,
                            epochs: int = 8, batch_size: int = 64):
    """
    Train MobileNetV2 to predict the source label, and evaluate on the test split.

    Returns a dictionary of test-set metrics. Balanced accuracy and AUROC are the
    headline figures because the within-class tasks are imbalanced.
    """
    tr = frame[frame["split"] == "train"]
    va = frame[frame["split"] == "val"]
    te = frame[frame["split"] == "test"]

    n_classes = int(frame[label_col].nunique())
    print(f"\n--- {title} ---")
    print(f"  train {len(tr):4d} | val {len(va):4d} | test {len(te):4d} "
          f"| {n_classes} source classes")
    print(f"  test balance: {dict(te[label_col].value_counts().sort_index())}")

    loaders = {
        "train": DataLoader(SourceDataset(tr, label_col, train_tf),
                            batch_size=batch_size, shuffle=True, num_workers=4,
                            pin_memory=True),
        "val":   DataLoader(SourceDataset(va, label_col, eval_tf),
                            batch_size=batch_size, shuffle=False, num_workers=4),
        "test":  DataLoader(SourceDataset(te, label_col, eval_tf),
                            batch_size=batch_size, shuffle=False, num_workers=4),
    }

    model = models.mobilenet_v2(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.last_channel, n_classes)
    model = model.to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    best_val, best_state = 0.0, None
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in loaders["train"]:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            crit(model(x), y).backward()
            opt.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for x, y in loaders["val"]:
                x, y = x.to(DEVICE), y.to(DEVICE)
                correct += (model(x).argmax(1) == y).sum().item()
                total += y.size(0)
        acc = correct / total
        if acc > best_val:
            best_val = acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"    epoch {ep}/{epochs}  val acc {acc:.4f}")

    model.load_state_dict(best_state)
    model.eval()

    probs, preds, labels = [], [], []
    with torch.no_grad():
        for x, y in loaders["test"]:
            out = torch.softmax(model(x.to(DEVICE)), dim=1).cpu()
            probs.append(out.numpy())
            preds.append(out.argmax(1).numpy())
            labels.append(y.numpy())
    probs  = np.concatenate(probs)
    preds  = np.concatenate(preds)
    labels = np.concatenate(labels)

    acc  = float((preds == labels).mean())
    bal  = float(balanced_accuracy_score(labels, preds))
    try:
        auroc = float(roc_auc_score(labels, probs[:, 1]) if n_classes == 2
                      else roc_auc_score(labels, probs, multi_class="ovr"))
    except Exception:
        auroc = float("nan")
    majority = float(pd.Series(labels).value_counts(normalize=True).max())

    print(f"  TEST accuracy {acc:.4f} | balanced {bal:.4f} | AUROC {auroc:.4f}")
    print(f"  majority-class baseline {majority:.4f}")

    return {"task": title, "n_test": len(labels), "accuracy": round(acc, 4),
            "balanced_accuracy": round(bal, 4), "auroc": round(auroc, 4),
            "majority_baseline": round(majority, 4)}

## 5. Global Source Classifier.

Train a classifier to separate CheXpert from the remaining sources across the
whole corpus.

This is the test as literally requested, and it is expected to succeed almost
perfectly. That success is not by itself evidence about the diagnostic models,
because CheXpert membership and the Pneumonia label are the same partition of the
corpus. The result establishes an upper bound: the sources are separable, so a
diagnostic model has the opportunity to exploit them.

#### Rationale:
- Answer the request as put, before qualifying it.
- Establish that the sources are distinguishable at all, which the within-class
  test then localises.
- Report the majority-class baseline alongside, so that a high accuracy is read
  against what a trivial classifier would achieve.
- State the entanglement with class explicitly rather than leaving it implicit.

In [5]:
global_task = corpus.copy()
global_task["source_label"] = (global_task["source_true"] == "chexpert").astype(int)

results = []
results.append(train_source_classifier(
    global_task, "source_label", "Global: CheXpert versus all other sources"))

print()
print("Interpretation: CheXpert membership and the Pneumonia label partition the")
print("corpus identically, so this figure is an upper bound on source separability")
print("and not a measure of confounding in the diagnostic task.")


--- Global: CheXpert versus all other sources ---
  train 2255 | val  481 | test  480 | 2 source classes
  test balance: {0: np.int64(320), 1: np.int64(160)}
    epoch 1/8  val acc 0.9979
    epoch 2/8  val acc 0.9979
    epoch 3/8  val acc 1.0000
    epoch 4/8  val acc 1.0000
    epoch 5/8  val acc 1.0000
    epoch 6/8  val acc 1.0000
    epoch 7/8  val acc 1.0000
    epoch 8/8  val acc 1.0000
  TEST accuracy 1.0000 | balanced 1.0000 | AUROC 1.0000
  majority-class baseline 0.6667

Interpretation: CheXpert membership and the Pneumonia label partition the
corpus identically, so this figure is an upper bound on source separability
and not a measure of confounding in the diagnostic task.


## 6. Within-Class Source Classifier.

Train a classifier to separate TBX11K from the smaller sources, separately within
the Normal class and within the Tuberculosis class.

Here the diagnosis is fixed and only the source varies, so any success is
attributable to acquisition characteristics alone. This is the measurement that
determines whether the confound is real rather than merely possible.

#### Rationale:
- Hold the diagnosis constant so that source and class are no longer entangled.
- Run the test in both classes that span multiple sources, so that the finding
  does not rest on one class.
- Compare against the majority-class baseline, since roughly ninety per cent of
  Normal images come from a single source.
- Use balanced accuracy and AUROC as the headline figures for the same reason.

In [6]:
for cls in ("Normal", "TB"):
    sub = corpus[corpus["Class"] == cls].copy()
    sub["source_label"] = (sub["source_true"] != "tbx11k").astype(int)
    if sub.groupby("split")["source_label"].nunique().min() < 2:
        print(f"[Skip] {cls}: a split contains only one source.")
        continue
    results.append(train_source_classifier(
        sub, "source_label", f"Within {cls}: TBX11K versus Montgomery/Shenzhen/DA/DB"))

source_df = pd.DataFrame(results)
source_df.to_csv(TABLE_DIR / "source_classifier_results.csv", index=False)
print()
print(source_df.to_string(index=False))


--- Within Normal: TBX11K versus Montgomery/Shenzhen/DA/DB ---
  train  751 | val  161 | test  160 | 2 source classes
  test balance: {0: np.int64(150), 1: np.int64(10)}
    epoch 1/8  val acc 0.9565
    epoch 2/8  val acc 0.9006
    epoch 3/8  val acc 0.9938
    epoch 4/8  val acc 1.0000
    epoch 5/8  val acc 1.0000
    epoch 6/8  val acc 1.0000
    epoch 7/8  val acc 1.0000
    epoch 8/8  val acc 1.0000
  TEST accuracy 1.0000 | balanced 1.0000 | AUROC 1.0000
  majority-class baseline 0.9375

--- Within TB: TBX11K versus Montgomery/Shenzhen/DA/DB ---
  train  751 | val  161 | test  160 | 2 source classes
  test balance: {0: np.int64(125), 1: np.int64(35)}
    epoch 1/8  val acc 0.8323
    epoch 2/8  val acc 0.9565
    epoch 3/8  val acc 0.9441
    epoch 4/8  val acc 0.9876
    epoch 5/8  val acc 0.9876
    epoch 6/8  val acc 0.9876
    epoch 7/8  val acc 0.9876
    epoch 8/8  val acc 0.9876
  TEST accuracy 0.9875 | balanced 0.9817 | AUROC 0.9986
  majority-class baseline 0.7812

   

## 7. Confound-Free Evaluation.

Evaluate the eight trained checkpoints on a subset in which the acquisition source
is held constant: Tuberculosis against Normal, both drawn from TBX11K alone.

Within this subset a model cannot separate the classes by identifying the dataset
of origin, because every image comes from the same one. Performance here is
therefore attributable to radiographic content. Comparing it against the
three-class result indicates how much of the headline figure survives when the
opportunity to exploit source is removed.

The comparison is not exact: this is a two-class problem and the original is a
three-class one, so chance performance differs. It is reported as a
confound-controlled check rather than as a like-for-like accuracy.

#### Rationale:
- Remove the confound by construction rather than adjusting for it statistically.
- Use the existing checkpoints without retraining, so the models evaluated are
  exactly those reported in the paper.
- Restrict the decision to the two classes present, since Pneumonia cannot appear
  in a TBX11K-only subset.
- Report the subset size, as it is smaller than the full test split and the
  interval is correspondingly wider.

In [7]:
def build_model(model_name: str, num_classes: int) -> nn.Module:
    if model_name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=None)
        m.classifier[1] = nn.Linear(m.last_channel, num_classes)
    elif model_name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "victorio":
        m = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(32 * 4 * 4, num_classes),
        )
    else:
        raise ValueError(model_name)
    return m


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z**2 / n
    c = p + z**2 / (2 * n)
    s = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return p, max(0.0, (c - s) / d), min(1.0, (c + s) / d)


subset = corpus[(corpus["split"] == "test") &
                (corpus["source_true"] == "tbx11k") &
                (corpus["Class"].isin(["TB", "Normal"]))].reset_index(drop=True)

print(f"Confound-free subset: {len(subset)} images, all TBX11K")
print(subset["Class"].value_counts().to_string())
print()

TB_IDX, NORMAL_IDX = CLASS_TO_INDEX["TB"], CLASS_TO_INDEX["Normal"]
truth = subset["Class"].map(CLASS_TO_INDEX).to_numpy()

cf_rows = []
for model_name in ("mobilenet_v2", "efficientnet_b0", "resnet50", "victorio"):
    for regime in ("imagenet", "cxr"):
        ckpt = CHECKPOINTS / f"{model_name}_{regime}_best.pt"
        if not ckpt.exists():
            continue

        mean, std = ((IMAGENET_MEAN, IMAGENET_STD) if regime == "imagenet"
                     else (CXR_MEAN, CXR_STD))
        _, tf = build_shared_transforms(img_size=IMG_SIZE, mean=mean, std=std)

        model = build_model(model_name, len(CLASS_TO_INDEX))
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        model = model.to(DEVICE).eval()

        loader = DataLoader(SourceDataset(
            subset.assign(_lbl=truth), "_lbl", tf),
            batch_size=64, shuffle=False, num_workers=4)

        preds = []
        with torch.no_grad():
            for x, _ in loader:
                logits = model(x.to(DEVICE))
                # Restrict the decision to the two classes present in the subset.
                two = logits[:, [TB_IDX, NORMAL_IDX]]
                choice = two.argmax(1).cpu().numpy()
                preds.append(np.where(choice == 0, TB_IDX, NORMAL_IDX))
        preds = np.concatenate(preds)

        acc, lo, hi = wilson(int((preds == truth).sum()), len(truth))
        tb_hits = int(((preds == TB_IDX) & (truth == TB_IDX)).sum())
        tb_n    = int((truth == TB_IDX).sum())
        tb_rec, tb_lo, tb_hi = wilson(tb_hits, tb_n)

        print(f"  {model_name:16s} {regime:9s} acc {acc:.4f} [{lo:.4f}, {hi:.4f}] "
              f"| TB recall {tb_rec:.4f} [{tb_lo:.4f}, {tb_hi:.4f}]")

        cf_rows.append({"model": model_name, "regime": regime,
                        "n": len(truth), "accuracy": round(acc, 4),
                        "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                        "tb_recall": round(tb_rec, 4)})

cf_df = pd.DataFrame(cf_rows)
cf_df.to_csv(TABLE_DIR / "confound_free_evaluation.csv", index=False)
print()
print("Saved:", (TABLE_DIR / "confound_free_evaluation.csv").relative_to(PROJECT_ROOT))

Confound-free subset: 275 images, all TBX11K
Class
Normal    150
TB        125

  mobilenet_v2     imagenet  acc 0.9891 [0.9684, 0.9963] | TB recall 0.9840 [0.9435, 0.9956]
  mobilenet_v2     cxr       acc 0.8982 [0.8568, 0.9286] | TB recall 0.8720 [0.8022, 0.9197]
  efficientnet_b0  imagenet  acc 0.9745 [0.9484, 0.9876] | TB recall 0.9600 [0.9098, 0.9828]
  efficientnet_b0  cxr       acc 0.9418 [0.9076, 0.9639] | TB recall 0.9200 [0.8590, 0.9560]
  resnet50         imagenet  acc 0.9855 [0.9632, 0.9943] | TB recall 0.9840 [0.9435, 0.9956]
  resnet50         cxr       acc 0.9636 [0.9344, 0.9801] | TB recall 0.9680 [0.9206, 0.9875]
  victorio         imagenet  acc 0.8364 [0.7881, 0.8754] | TB recall 0.7120 [0.6272, 0.7841]
  victorio         cxr       acc 0.8327 [0.7841, 0.8722] | TB recall 0.7280 [0.6441, 0.7983]

Saved: reports/tables/confound_free_evaluation.csv


## 8. Summary.

Bring the three measurements together and state what each supports.

#### Rationale:
- Separate what each test establishes, since they answer different questions.
- Give the figures the paper should quote, in one place.
- Record the limitation that remains after the analysis.
- Avoid drawing a conclusion the measurements do not support.

In [8]:
print("=" * 74)
print("SOURCE CONFOUND: SUMMARY")
print("=" * 74)

print()
print("1. Pixel signature (no model involved)")
sig = pd.read_csv(TABLE_DIR / "source_pixel_signature.csv")
print(sig.to_string(index=False))

print()
print("2. Source classifiers")
print(source_df.to_string(index=False))

print()
print("3. Confound-free evaluation, TBX11K-only TB versus Normal")
print(cf_df.to_string(index=False))

print()
print("=" * 74)
print("The global classifier separates CheXpert from the rest, but CheXpert")
print("membership and the Pneumonia label are the same partition, so that figure")
print("bounds separability rather than measuring confounding.")
print()
print("The within-class classifiers hold the diagnosis fixed. Their scores,")
print("read against the majority-class baseline, indicate whether acquisition")
print("source is recoverable independently of pathology.")
print()
print("The confound-free evaluation removes the opportunity to exploit source")
print("altogether. It is a two-class problem and so not directly comparable with")
print("the three-class headline, but it shows what the models achieve when every")
print("image shares an acquisition source.")
print("=" * 74)

SOURCE CONFOUND: SUMMARY

1. Pixel signature (no model involved)
 class  same_source_mean  diff_source_mean  difference  t_statistic  n_same  n_diff
Normal            0.7114            0.6193      0.0922        153.1  486651   87405
    TB            0.6040            0.5744      0.0295         67.2  341804  232252

2. Source classifiers
                                                  task  n_test  accuracy  balanced_accuracy  auroc  majority_baseline
             Global: CheXpert versus all other sources     480    1.0000             1.0000 1.0000             0.6667
Within Normal: TBX11K versus Montgomery/Shenzhen/DA/DB     160    1.0000             1.0000 1.0000             0.9375
    Within TB: TBX11K versus Montgomery/Shenzhen/DA/DB     160    0.9875             0.9817 0.9986             0.7812

3. Confound-free evaluation, TBX11K-only TB versus Normal
          model   regime   n  accuracy  ci_low  ci_high  tb_recall
   mobilenet_v2 imagenet 275    0.9891  0.9684   0.9963      0

## Appendix: What This Analysis Does and Does Not Establish.

**Establishes.** Whether acquisition source is recoverable from the images, both
in raw pixels and by a trained classifier, with the diagnosis held constant.
Whether the trained checkpoints retain their performance on a subset where source
cannot be exploited.

**Does not establish.** That the diagnostic models do or do not use source
information. Demonstrating that a cue is available is not the same as
demonstrating that a particular model relies on it. Attribution methods speak to
this indirectly, and the confound-free evaluation bounds it, but neither settles it.

**Remains a limitation.** The corpus cannot be made source-balanced without a
Pneumonia set from TBX11K or a tuberculosis set from CheXpert, neither of which
exists. Domain-adversarial training, following Ganin et al. (2016), would be the
appropriate remedy and is beyond the scope of this paper. The result should be
read as an upper bound on clinical performance, and the confound stated as the
principal limitation rather than defended against.